<a href="https://colab.research.google.com/github/efthymios19/HYDRUS_GoogleColab/blob/main/SWRC/swrc_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Παράδειγμα προσαρμογής πειραματικών δεδομένων χαρακτηριστικής καμπύλης εδαφικής υγρασίας με χρήση των μοντέλων Brooks - Corey και Van Genuchten

Τα πειραματικά δεδομένα προέρχονται από τις μεθόδους της φυγοκέντρου και του διαλύματος άλατος συνδυαστικά.

#### Εισαγωγή/εγκατάσταση βιβλιοθηκών – πακέτων

In [ ]:
# Εγκατάσταση των απαιτούμενων βιβλιοθηκων

!pip install  unsatfit

In [ ]:
import math
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

import unsatfit as uns

#### Επεξεργασία δεδομένων δείγματος 0 – 10 cm

In [ ]:
W=[74.87, 67.61, 65.01, 63.80, 62.70, 61.68, 60.85, 60.09, 58.326, 56.152] # βάρη που προκύπτουν 
# από τη ζύγιση των δειγμάτων σε κάθε γωνιακή ταχύτητα περιστροφής (rpm) της φυγοκέντρου
# και σε κάθε μοριακότητα του αλατούχου διαλύματος 
# οι μονάδες είναι g
# το πρώτο βάρος αντιστοιχεί στον κορεσμό

In [ ]:
War=np.array(W) # μετατροπή της λίστας W σε πίνακα numpy
print(np.shape(War)) # εμφάνιση των διαστάσεων του πίνακα σε μορφή (γραμμές, στήλες)
print(War) # εμφάνιση των βαρών τoυ δείγματος

In [ ]:
War=War-10.26 # αφαιρούμε το βάρος του liner (10.26 g) από όλα τα βάρη των δειγμάτων
print(War) # εμφάνιση των αφαιρεμένων βαρών

#### Υπολογισμός φαινόμενης πυκνότητας και πορώδους

In [ ]:
Ms=37.314 # μάζα του ξηρού δείγματος (g) 

In [ ]:
height=2.8
rad=2.3

Vt=height*math.pi*rad**2 # όγκος δείγματος (cm^3)

In [ ]:
ρb=Ms/Vt # φαινόμενη πυκνότητα (g/cm^3)
print(ρb) # εμφάνιση της φαινόμενης/φαινομενικής πυκνότητας

In [ ]:
Porosity=(War[0]-Ms)/Vt # υπολογισμός της πορώδους του δείγματος
print(Porosity) # εμφάνιση της πορώδους του δείγματος

#### Υπολογισμός περιεκτικότητας σε νερό κατά μάζα

In [ ]:
theta_g=(War-Ms)/Ms # υπολογισμός της υγρασίας (θ) του δείγματος κατά μάζα στην 
# εκάστοτε ζύγιση, Mw/Ms, όπου Mw είναι η μάζα του νερού που περιέχεται στο δείγμα και 
# Ms η μάζα του ξηρού δείγματος
print(theta_g) # εμφάνιση της υγρασίας (θ) του δείγματος κατά μάζα

#### Υπολογισμός περιεκτικότητας σε νερό κατ' όγκο

In [ ]:
theta_v=theta_g*ρb # υπολογισμός της υγρασίας (θ) του δείγματος κατά όγκο στην εκάστοτε ζύγιση, θv=θg*ρb,
# όπου θg είναι η υγρασία κατά μάζα και ρb η φαινόμενη πυκνότητα του δείγματος (θεώροντας ότι η πυκνότητα 
# του νερού είναι ίση με ένα)
print(theta_v) # εμφάνιση της υγρασίας (θ) του δείγματος κατά όγκο

#### Υπολογισμός δυναμικών από τις γωνιακές ταχύτητες και τις μοριακότητες του διαλύματος

$$
h_i = \beta(\rho_w-\rho_g)\omega_i^2
\left(r_b^2-r_t^2\right)
$$

όπου:

- $\rho_w$: πυκνότητα του νερού των πόρων,
  ίση με $1.0\,\mathrm{kg/L}$.
- $\rho_g$: πυκνότητα του αέρα των πόρων,
  η οποία θεωρείται αμελητέα ($\rho_g \approx 0$).
- $\omega_i$: ταχύτητα περιστροφής της φυγοκέντρου
  στο βήμα $i$, σε $\mathrm{rpm}$.
- $r_b$: εξωτερική ακτίνα από τον άξονα περιστροφής
  έως την ελεύθερη επιφάνεια διαρροής του νερού,
  σε $\mathrm{m}$.
- $r_t$: εσωτερική ακτίνα από τον άξονα περιστροφής
  έως το ανώτερο σημείο του εδαφικού δείγματος,
  σε $\mathrm{m}$.
- $\beta$: συντελεστής μετατροπής μονάδων,
  με τιμή $0.00553$ για τις παραπάνω μονάδες.

In [ ]:
S=[0,500,800,1000,1500,2000,2500,3000] # γωνιακές ταχύτητες σε rpm που υποβλήθηκε
# σε φυγοκέντρηση το δείγμα (rpm)

In [ ]:
Sar=np.array(S) # μετατροπή της λίστας των γωνιακών ταχυτήτων
# σε numpy πίνακα
print(np.shape(Sar)) # εμφάνιση των διαστάσεων του πίνακα σε μορφή (γραμμές, στήλες)
print(Sar) # εμφάνιση των rpm

In [ ]:
b=0.00553 # αδιάστατη παράμετρος

In [ ]:
rb=(9+12.7+0.4)/100 # εξωτερική ακτίνα από τον άξονα περιστροφής
# έως την ελεύθερη επιφάνεια διαρροής του νερού (cm --> m)


In [ ]:
rt=((9+12.7)-2.7)/100 # εσωτερική ακτίνα από τον άξονα περιστροφής
# έως το ανώτερο σημείο του εδαφικού δείγματος (cm --> m)

In [ ]:
# Δημιούργία βρόχου επανάληψης για να τυπώσουμε όλες 
# τις τιμές της εξίσωσης με βάση τις τιμές στον 
# numpy πίνακα

for i in Sar:
    a=b*i**2*(rb**2-rt**2)
    print(a)

In [ ]:
psi=b*Sar**2*(rb**2-rt**2) # υπολογισμός πιέσεων για όλες τις τιμές του πίνακα

In [ ]:
psi=np.append(psi,[2000,8000],axis=0) # προσθήκη των πιέσεων που προκύπτουν από τη μοριακότητα του διαλύματος

In [ ]:
psi_cm=psi/0.0980638 # μετατροπή των μονάδων από kPa σε cm (P = ρ g h)

#### Προσαρμογή μοντέλων Van Genuchten και Mualem

Για τα δεδομένα της χαρακτηριστικής καμπύλης, η βιβλιοθήκη unsatfit ελαχιστοποιεί τις τετραγωνικες διαφορές μεταξύ των μετρημένων και προσομοιωμένων τιμών της περικτικότητας σε νερό: 
$$ \min_{\mathbf p}\sum_i
\left[\theta_{\mathrm{model}}(h_i;\mathbf p)
-\theta_{\mathrm{measured},i}\right]^2 $$

In [ ]:
# Προσαρμογή μοντέλου Van Genuchten 
h_data = np.asarray(psi_cm, dtype=float) # μετονομασία των πινάκων σε h_data και theta_data
theta_data = np.asarray(theta_v, dtype=float)

assert h_data.shape == theta_data.shape # επιβεβαίωση ότι οι δύο πίνακες έχουν το ίδιο σχήμα
assert np.all(np.isfinite(h_data)) # επιβεβαίωση ότι όλες οι τιμές στον πίκανα h_data είναι πεπερασμένες 
assert np.all(np.isfinite(theta_data)) # επιβεβαίωση ότι όλες οι τιμές στον πίκανα theta_data είναι πεπερασμένες 
assert np.all(h_data >= 0) # επιβεβαίωση ότι τα δεδομένα του πίνακα h_data είναι μεγαλύτερα ή ίσα από 0
assert np.all((theta_data >= 0) & (theta_data <= 1)) # επιβεβαίωση ότι τα δεδομένα του πίνακα theta_data είναι μεγαλύτερα ή ίσα από 0 
# και ταυτόχρονα μικρότερα ή ίσα με 1

fit_vg = uns.Fit() # Εκκίνησης του πακέτου unsatfit
fit_vg.set_model("VG", const=["q=1"]) # Επιλογή μοντέλου , m = 1 - 1 / n [q=1]
fit_vg.swrc = (h_data, theta_data) # Επιλογή δεδομένων

# Αρχικές εκτιμήσεις παραμέτρων
fit_vg.ini = (max(theta_data), 0, *fit_vg.get_init()) # λήψη αρχικών εκτιμήσεων
fit_vg.optimize() # βελτιστοποίηση παραμέτρων

if not fit_vg.success:
    raise RuntimeError(fit_vg.message) # αν δεν είναι επιτυχημένη η βελτισοποίηση τυπώνει μήνυμα

# UNSATFIT επίστεφει το m oπότε πρέπει να υπολογίσουμε το n
theta_s, theta_r, alpha, m_vg = fit_vg.fitted # ανάκτηση βελτιστοποιημένων παραμέτρων
n_vg = 1 / (1 - m_vg)

print(f"theta_r = {theta_r:.5f}")
print(f"theta_s = {theta_s:.5f}")
print(f"alpha   = {alpha:.6f} cm⁻¹")
print(f"n       = {n_vg:.5f}")
print(f"m       = {m_vg:.5f}")
print(f"R²      = {fit_vg.r2_ht:.5f}")

In [ ]:
# Αντίστοιχη διαδικασία για το μοντέλο Brooks Corey

h_data = np.asarray(psi_cm, dtype=float)
theta_data = np.asarray(theta_v, dtype=float)

fit_bc = uns.Fit()
fit_bc.set_model("BC", const=[])
fit_bc.swrc = (h_data, theta_data)


fit_bc.ini = (max(theta_data), 0, *fit_bc.get_init())
fit_bc.optimize()

if not fit_bc.success:
    raise RuntimeError(fit_bc.message)

theta_s_bc, theta_r_bc, hb_bc, lambda_bc = fit_bc.fitted

print(f"theta_r = {theta_r_bc:.5f}")
print(f"theta_s = {theta_s_bc:.5f}")
print(f"hb      = {hb_bc:.5f} cm")
print(f"lambda  = {lambda_bc:.5f}")
print(f"R²      = {fit_bc.r2_ht:.5f}")

In [ ]:
# Απεικόνιση της προσαρμογής μεταξύ παρατηρημένων τιμών και τιμών των παραμετρικών μοντέλων

# Ανάκτηση των παραμέτρων κάθε μοντέλου

ts_vg, tr_vg, alpha_vg, m_vg = fit_vg.fitted
n_vg = 1 / (1 - m_vg)

ts_bc, tr_bc, hb_bc, lambda_bc = fit_bc.fitted

# Include exactly zero and sample the remaining range densely
h_positive = h_data[h_data > 0] # αποκλείουμε το 0

h_plot = np.r_[
    0.0,
    np.geomspace(h_positive.min() / 1000, h_data.max(), 1000) # δημιούργια 1000 θετικών τιμών με λογαριθμική κατανομή από τη
     # μικροτέρη θετική τιμή έως τη μεγαλύτερη
]

# van Genuchten
theta_vg_plot = tr_vg + (ts_vg - tr_vg) * (
    1 + (alpha_vg * h_plot)**n_vg
)**(-m_vg) # υπολογισμός της περιεκτικότητας σε νερό σε κάθε τιμή δυναμικού που προκύπτει από το h_plot

# Brooks–Corey: avoid division by zero
Se_bc = np.ones_like(h_plot) # πίνακας ίδιου μεγέθους με h_plot με μονάδες (Se=1)
draining = h_plot > hb_bc #  λογική μάσκα όπου το δυναμικό υπερβαίνει το δυναμικό εισαγωγής αέρα (hb-bc)
Se_bc[draining] = (hb_bc / h_plot[draining])**lambda_bc # υπολογίζεται ο βαθμός κορεσμός στα σημεία αποστράγγισης

theta_bc_plot = tr_bc + (ts_bc - tr_bc) * Se_bc # ο κορεσμός μετατρέπεται σε περιεκτικότητα σε νερό

# Combined diagram
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    h_data, theta_data,
    color="black", s=45, zorder=4, label="Measured"
)

ax.plot(
    h_plot, theta_vg_plot,
    color="royalblue", linewidth=2, label="van Genuchten"
)

ax.plot(
    h_plot, theta_bc_plot,
    color="darkorange", linewidth=2,
    linestyle="--", label="Brooks–Corey"
)

ax.set_xscale("symlog", linthresh=10)
ax.set_xlim(0, h_data.max() * 1.1)
ax.set_ylim(bottom=0)

ax.set_xlabel("Μύζηση [cm]")
ax.set_ylabel("Περιεκτικότητα σε νερό [cm³/cm³]")
ax.set_title("Προσαρμογή μοντέλων VG και BC σε πειραματικά δεδομένα")
ax.grid(True, which="both", alpha=0.25)
ax.legend()

fig.tight_layout()
#fig.savefig("SWRC_VG_BC.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Το ίδο γράφημε με pF στον άξονα x

positive_data = h_data > 0
positive_curve = h_plot > 0

pf_data = np.log10(h_data[positive_data])
pf_curve = np.log10(h_plot[positive_curve])

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    pf_data, theta_data[positive_data],
    color="black", s=45, zorder=4, label="Measured"
)

ax.plot(
    pf_curve, theta_vg_plot[positive_curve],
    color="royalblue", linewidth=2,
    label="van Genuchten"
)

ax.plot(
    pf_curve, theta_bc_plot[positive_curve],
    color="darkorange", linewidth=2,
    linestyle="--", label="Brooks–Corey"
)

ax.set_xlabel(r"pF = $\log_{10}(|h|\,[\mathrm{cm}])$")
ax.set_ylabel("Περιεκτικότητα σε νερό [cm³/cm³]")
ax.set_title("Χαρακτηριστική καμπύλη εδαφικής υγρασίας")
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.25)
ax.legend()

fig.tight_layout()
#fig.savefig("SWRC_VG_BC_pF.png", dpi=300, bbox_inches="tight")
plt.show()

#### Επαναλάβετε τη διαδικάσια για τα δεδομένα του δείγματος 80 – 90 cm και συγκρίνετε με το δείγμα 0 – 10 cm

Βάρη: [105.15, 103.43, 102.73, 102.16, 101.29, 100.21, 99.21, 98.32, 97.38, 95]

height=3 (μεγαλύτερο ύψος κυλίνδρου)


